# 04 — Final Dataset Summary

**Project:** Ontology-Guided Hypothesis Generation Using LLMs and Topic Modeling in mHealth Research

This notebook provides a complete summary of the dataset preparation pipeline,
including collection, merging, cleaning, relevance filtering, and final statistics.

**Date:** 2026-08-29

In [1]:
import pandas as pd
import numpy as np
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
REPORTS_DIR = os.path.join(PROJECT_ROOT, 'reports', 'summaries')
os.makedirs(REPORTS_DIR, exist_ok=True)

print('Setup complete.')

Setup complete.


## 1. Load All Datasets

In [2]:
# Load all datasets for comparison
df_original = pd.read_csv(os.path.join(RAW_DIR, 'mhealth_pubmed_original_491.csv'))

# Find expanded file
import glob
expanded_files = sorted(glob.glob(os.path.join(RAW_DIR, 'pubmed_expanded_*.csv')))
df_expanded = pd.read_csv(expanded_files[-1]) if expanded_files else pd.DataFrame()

df_cleaned = pd.read_csv(os.path.join(PROCESSED_DIR, 'mhealth_final_cleaned_dataset.csv'))
df_annotated = pd.read_csv(os.path.join(PROCESSED_DIR, 'mhealth_relevance_annotated.csv'))
df_relevant = pd.read_csv(os.path.join(PROCESSED_DIR, 'mhealth_relevant_dataset.csv'))
df_low_rel = pd.read_csv(os.path.join(PROCESSED_DIR, 'mhealth_low_relevance.csv'))

print('All datasets loaded successfully.')

All datasets loaded successfully.


## 2. Pipeline Summary

In [3]:
# Calculate key metrics
original_count = len(df_original)
expanded_count = len(df_expanded)
total_before_dedup = original_count + expanded_count
cleaned_count = len(df_cleaned)
duplicates_removed = total_before_dedup - cleaned_count  # Approximate (includes missing/short)

# Read merge report for exact numbers
merge_report_path = os.path.join(REPORTS_DIR, 'merge_report.txt')
merge_report = ''
if os.path.exists(merge_report_path):
    with open(merge_report_path, 'r', encoding='utf-8') as f:
        merge_report = f.read()

annotated_count = len(df_annotated)
relevant_count = len(df_relevant)
low_rel_count = len(df_low_rel)

print('=' * 60)
print('COMPLETE PIPELINE SUMMARY')
print('=' * 60)
print(f'\n1. INITIAL DATASET')
print(f'   Papers initially available: {original_count}')
print(f'   Source: PubMed (original collection)')
print(f'   Columns: {df_original.columns.tolist()}')
print(f'\n2. EXPANDED COLLECTION')
print(f'   Additional papers collected: {expanded_count}')
print(f'   Source: PubMed (10 diverse mHealth queries, 2018-2026)')
print(f'   Total before deduplication: {total_before_dedup}')
print(f'\n3. MERGE & DEDUPLICATION')
print(f'   After merge and dedup: {cleaned_count}')
print(f'   Records removed: {total_before_dedup - cleaned_count}')
print(f'   (includes PMID duplicates, title duplicates, missing abstracts, short abstracts)')
print(f'\n4. RELEVANCE FILTERING')
print(f'   Papers analyzed: {annotated_count}')
if 'relevance_category' in df_annotated.columns:
    cat_counts = df_annotated['relevance_category'].value_counts()
    for cat in ['Highly relevant', 'Relevant', 'Possibly relevant', 'Low relevance']:
        count = cat_counts.get(cat, 0)
        pct = 100 * count / annotated_count
        print(f'   {cat}: {count} ({pct:.1f}%)')
print(f'\n5. FINAL DATASET')
print(f'   Papers retained: {relevant_count}')
print(f'   Papers removed (low relevance): {low_rel_count}')
print(f'   Columns: {df_relevant.columns.tolist()}')

COMPLETE PIPELINE SUMMARY

1. INITIAL DATASET
   Papers initially available: 491
   Source: PubMed (original collection)
   Columns: ['pmid', 'title', 'abstract', 'year', 'mesh_terms']

2. EXPANDED COLLECTION
   Additional papers collected: 2177
   Source: PubMed (10 diverse mHealth queries, 2018-2026)
   Total before deduplication: 2668

3. MERGE & DEDUPLICATION
   After merge and dedup: 2628
   Records removed: 40
   (includes PMID duplicates, title duplicates, missing abstracts, short abstracts)

4. RELEVANCE FILTERING
   Papers analyzed: 2628
   Highly relevant: 234 (8.9%)
   Relevant: 593 (22.6%)
   Possibly relevant: 923 (35.1%)
   Low relevance: 878 (33.4%)

5. FINAL DATASET
   Papers retained: 1750
   Papers removed (low relevance): 878
   Columns: ['pmid', 'title', 'abstract', 'year', 'mesh_terms', 'journal', 'authors', 'doi', 'document', 'cleaned_text', 'source', 'keyword_score', 'matched_keywords', 'mesh_score', 'matched_mesh', 'tfidf_score', 'relevance_score', 'relevance_ca

## 3. Final Dataset Statistics

In [4]:
print('=' * 60)
print('FINAL DATASET STATISTICS')
print('=' * 60)
print(f'\nTotal papers: {len(df_relevant)}')
print(f'\nColumns ({len(df_relevant.columns)}):')
for col in df_relevant.columns:
    non_null = df_relevant[col].notna().sum()
    non_empty = (df_relevant[col].fillna('').astype(str).str.strip() != '').sum()
    print(f'  {col}: {non_empty} non-empty ({100*non_empty/len(df_relevant):.1f}%)')

print(f'\nYear distribution:')
df_relevant['year'] = df_relevant['year'].astype(str)
year_counts = df_relevant['year'].value_counts().sort_index()
for year, count in year_counts.items():
    if str(year).isdigit():
        print(f'  {year}: {count}')

print(f'\nAbstract length statistics (chars):')
abs_len = df_relevant['abstract'].str.len()
print(f'  Mean: {abs_len.mean():.0f}')
print(f'  Median: {abs_len.median():.0f}')
print(f'  Min: {abs_len.min():.0f}')
print(f'  Max: {abs_len.max():.0f}')

if 'relevance_score' in df_relevant.columns:
    print(f'\nRelevance score statistics:')
    print(f'  Mean: {df_relevant["relevance_score"].mean():.3f}')
    print(f'  Median: {df_relevant["relevance_score"].median():.3f}')
    print(f'  Min: {df_relevant["relevance_score"].min():.3f}')
    print(f'  Max: {df_relevant["relevance_score"].max():.3f}')

FINAL DATASET STATISTICS

Total papers: 1750

Columns (19):
  pmid: 1750 non-empty (100.0%)
  title: 1750 non-empty (100.0%)
  abstract: 1750 non-empty (100.0%)
  year: 1472 non-empty (84.1%)
  mesh_terms: 1180 non-empty (67.4%)
  journal: 1464 non-empty (83.7%)
  authors: 1470 non-empty (84.0%)
  doi: 1442 non-empty (82.4%)
  document: 1750 non-empty (100.0%)
  cleaned_text: 1750 non-empty (100.0%)
  source: 1750 non-empty (100.0%)
  keyword_score: 1750 non-empty (100.0%)
  matched_keywords: 1677 non-empty (95.8%)
  mesh_score: 1750 non-empty (100.0%)
  matched_mesh: 1161 non-empty (66.3%)
  tfidf_score: 1750 non-empty (100.0%)
  relevance_score: 1750 non-empty (100.0%)
  relevance_category: 1750 non-empty (100.0%)
  relevance_reason: 1750 non-empty (100.0%)

Year distribution:

Abstract length statistics (chars):
  Mean: 1722
  Median: 1651
  Min: 113
  Max: 4725

Relevance score statistics:
  Mean: 0.515
  Median: 0.484
  Min: 0.300
  Max: 1.000


## 4. Manual Sample Validation

In [5]:
# Generate random sample for manual validation
SAMPLE_SIZE = 50
RANDOM_SEED = 42

sample = df_relevant.sample(n=min(SAMPLE_SIZE, len(df_relevant)), random_state=RANDOM_SEED)

validation_cols = ['pmid', 'title', 'relevance_category', 'relevance_score', 'relevance_reason']
available_cols = [c for c in validation_cols if c in sample.columns]
validation_sample = sample[available_cols].copy()

# Save validation sample
validation_path = os.path.join(REPORTS_DIR, 'manual_validation_sample.csv')
validation_sample.to_csv(validation_path, index=False, encoding='utf-8')
print(f'Manual validation sample saved: {validation_path}')
print(f'Sample size: {len(validation_sample)} papers')
print(f'\nSample breakdown:')
if 'relevance_category' in validation_sample.columns:
    print(validation_sample['relevance_category'].value_counts().to_string())

print(f'\nFirst 10 papers in sample:')
print(validation_sample.head(10).to_string())

print(f'\n--- VALIDATION NOTES ---')
print('This sample is for manual review to verify automated relevance filtering.')
print('Reviewers should check whether each paper is genuinely related to mHealth.')
print('\nLIMITATIONS of automated relevance scoring:')
print('- Keyword-based: may miss papers using novel terminology')
print('- MeSH-dependent: ~40% papers lack MeSH terms (neutral scored)')
print('- TF-IDF: depends on vocabulary overlap with reference text')
print('- Borderline cases (score 0.3-0.4) most likely to be misclassified')
print('- Novel mHealth applications may be underscored')

Manual validation sample saved: E:\MAJOR PROJECT\reports\summaries\manual_validation_sample.csv
Sample size: 50 papers

Sample breakdown:
relevance_category
Possibly relevant    25
Relevant             21
Highly relevant       4

First 10 papers in sample:
          pmid                                                                                                                                                                                              title relevance_category  relevance_score                                                                            relevance_reason
1246  36788261                                                                                        Usability of a mobile application for health professionals in home care services: a user-centered approach.  Possibly relevant         0.382220      Weak keyword match (0.16); Good MeSH alignment (0.70); Good semantic similarity (0.36)
862   37256005                           A study to explore the use

## 5. Readiness Assessment

In [6]:
print('=' * 60)
print('READINESS ASSESSMENT FOR TF-IDF / TOPIC MODELING')
print('=' * 60)

checks = []

# Check 1: Dataset size
size_ok = len(df_relevant) >= 500
checks.append(('Dataset size >= 500 papers', size_ok, f'{len(df_relevant)} papers'))

# Check 2: All papers have abstracts
has_abstract = (df_relevant['abstract'].fillna('').str.strip() != '').all()
checks.append(('All papers have abstracts', has_abstract, ''))

# Check 3: All papers have titles
has_title = (df_relevant['title'].fillna('').str.strip() != '').all()
checks.append(('All papers have titles', has_title, ''))

# Check 4: document column exists
has_doc = 'document' in df_relevant.columns
checks.append(('Document column exists', has_doc, ''))

# Check 5: cleaned_text column exists
has_cleaned = 'cleaned_text' in df_relevant.columns
checks.append(('Cleaned text column exists', has_cleaned, ''))

# Check 6: No duplicate PMIDs
no_dupes = not df_relevant['pmid'].duplicated().any()
checks.append(('No duplicate PMIDs', no_dupes, ''))

# Check 7: Year coverage
n_years = df_relevant['year'].nunique()
year_ok = n_years >= 3
checks.append(('Covers 3+ years', year_ok, f'{n_years} years'))

# Check 8: Relevance filtered
rel_ok = 'relevance_score' in df_relevant.columns
checks.append(('Relevance filtering applied', rel_ok, ''))

all_ok = all(c[1] for c in checks)

for name, status, note in checks:
    icon = '✓' if status else '✗'
    extra = f' ({note})' if note else ''
    print(f'  [{icon}] {name}{extra}')

print(f'\n{"="*60}')
if all_ok:
    print('ALL CHECKS PASSED.')
    print('The dataset is ready for TF-IDF keyword extraction and topic modeling.')
    print(f'\nFinal dataset: data/processed/mhealth_relevant_dataset.csv')
    print(f'Papers: {len(df_relevant)}')
    print(f'\nKey columns for next stage:')
    print(f'  - document: Original title + abstract (for BERTopic/transformers)')
    print(f'  - cleaned_text: Cleaned text (for TF-IDF, LDA)')
    print(f'  - mesh_terms: PubMed MeSH headings (for ontology mapping)')
else:
    print('SOME CHECKS FAILED. Review issues above before proceeding.')
print('=' * 60)

READINESS ASSESSMENT FOR TF-IDF / TOPIC MODELING
  [✓] Dataset size >= 500 papers (1750 papers)
  [✓] All papers have abstracts
  [✓] All papers have titles
  [✓] Document column exists
  [✓] Cleaned text column exists
  [✓] No duplicate PMIDs
  [✓] Covers 3+ years (10 years)
  [✓] Relevance filtering applied

ALL CHECKS PASSED.
The dataset is ready for TF-IDF keyword extraction and topic modeling.

Final dataset: data/processed/mhealth_relevant_dataset.csv
Papers: 1750

Key columns for next stage:
  - document: Original title + abstract (for BERTopic/transformers)
  - cleaned_text: Cleaned text (for TF-IDF, LDA)
  - mesh_terms: PubMed MeSH headings (for ontology mapping)


## 6. Save Final Summary Report

In [7]:
# Generate final summary report
report = []
report.append('=' * 60)
report.append('FINAL DATASET SUMMARY REPORT')
report.append('Project: Ontology-Guided Hypothesis Generation Using LLMs')
report.append('         and Topic Modeling in mHealth Research')
report.append('Date: 2026-08-29')
report.append('=' * 60)
report.append('')
report.append('PIPELINE OVERVIEW')
report.append('-' * 40)
report.append(f'Step 1: Original dataset:          {original_count} papers')
report.append(f'Step 2: Expanded collection:        {expanded_count} papers')
report.append(f'Step 3: After merge + dedup:         {cleaned_count} papers')
report.append(f'Step 4: After relevance filtering:   {relevant_count} papers')
report.append(f'Step 5: Low relevance removed:       {low_rel_count} papers')
report.append(f'Step 6: Final relevant dataset:      {relevant_count} papers')
report.append('')
report.append('FINAL DATASET')
report.append('-' * 40)
report.append(f'File: data/processed/mhealth_relevant_dataset.csv')
report.append(f'Papers: {relevant_count}')
report.append(f'Columns: {df_relevant.columns.tolist()}')
report.append('')
report.append('READINESS: Dataset is ready for TF-IDF keyword extraction.')

report_text = '\n'.join(report)

report_path = os.path.join(REPORTS_DIR, 'final_dataset_summary.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_text)

print(report_text)
print(f'\nReport saved to: {report_path}')
print('\n\nDataset preparation and exploratory analysis are complete.')
print('The project is ready to move to TF-IDF keyword extraction.')

FINAL DATASET SUMMARY REPORT
Project: Ontology-Guided Hypothesis Generation Using LLMs
         and Topic Modeling in mHealth Research
Date: 2026-08-29

PIPELINE OVERVIEW
----------------------------------------
Step 1: Original dataset:          491 papers
Step 2: Expanded collection:        2177 papers
Step 3: After merge + dedup:         2628 papers
Step 4: After relevance filtering:   1750 papers
Step 5: Low relevance removed:       878 papers
Step 6: Final relevant dataset:      1750 papers

FINAL DATASET
----------------------------------------
File: data/processed/mhealth_relevant_dataset.csv
Papers: 1750
Columns: ['pmid', 'title', 'abstract', 'year', 'mesh_terms', 'journal', 'authors', 'doi', 'document', 'cleaned_text', 'source', 'keyword_score', 'matched_keywords', 'mesh_score', 'matched_mesh', 'tfidf_score', 'relevance_score', 'relevance_category', 'relevance_reason']

READINESS: Dataset is ready for TF-IDF keyword extraction.

Report saved to: E:\MAJOR PROJECT\reports\summar